In [1]:
import os

# os.mkdir('data')

In [2]:
import os
import shutil
import numpy as np
import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.layers import (
    Input,
    Add,
    Dropout,
    Dense,
    Activation,
    ZeroPadding2D,
    BatchNormalization,
    Flatten,
    Conv2D,
    AveragePooling2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import plot_model
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
    load_img,
    img_to_array,
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from IPython.display import SVG
import scipy.misc
from matplotlib.pyplot import imshow

%matplotlib inline

In [3]:
# Where all dataset is there
data_dir = "/home/datasets/flowers"

# Training data dir
training_dir = "data/Train"

# Test data dir
testing_dir = "data/Test"

# Ratio of training and testing data
train_test_ratio = 0.8


def split_dataset_into_test_and_train_sets(
    all_data_dir=data_dir,
    training_data_dir=training_dir,
    testing_data_dir=testing_dir,
    train_test_ratio=0.8,
):
    # Recreate testing and training directories

    if not os.path.exists(training_data_dir):
        os.mkdir(training_data_dir)

    if not os.path.exists(testing_data_dir):
        os.mkdir(testing_data_dir)

    num_training_files = 0
    num_testing_files = 0

    for subdir, dirs, files in os.walk(all_data_dir):
        category_name = os.path.basename(subdir)

        # print(category_name + " vs " + os.path.basename(all_data_dir))
        if category_name == os.path.basename(all_data_dir):
            continue

        training_data_category_dir = training_data_dir + "/" + category_name
        testing_data_category_dir = testing_data_dir + "/" + category_name

        # creating subdir for each sub category
        if not os.path.exists(training_data_category_dir):
            os.mkdir(training_data_category_dir)

        if not os.path.exists(testing_data_category_dir):
            os.mkdir(testing_data_category_dir)

        file_list = glob.glob(os.path.join(subdir, "*.jpg"))

        # print(os.path.join(all_data_dir, subdir))
        print(str(category_name) + " has " + str(len(files)) + " images")
        random_set = np.random.permutation((file_list))
        # copy percentage of data from each category to train and test directory
        train_list = random_set[: round(len(random_set) * (train_test_ratio))]
        test_list = random_set[-round(len(random_set) * (1 - train_test_ratio)) :]

        for lists in train_list:
            shutil.copy(lists, training_data_dir + "/" + category_name + "/")
            num_training_files += 1

        for lists in test_list:
            shutil.copy(lists, testing_data_dir + "/" + category_name + "/")
            num_testing_files += 1

    print("Processed " + str(num_training_files) + " training files.")
    print("Processed " + str(num_testing_files) + " testing files.")

In [4]:
split_dataset_into_test_and_train_sets()

Processed 0 training files.
Processed 0 testing files.


In [5]:
# Number of classes in dataset
num_classes = 5


def get_model():
    # Get base model
    # Here we are using ResNet50 as base model
    base_model = ResNet50(weights="imagenet", include_top=False)

    # As we are using ResNet model only for feature extraction and not adjusting the weights
    # we freeze the layers in base model
    for layer in base_model.layers:
        layer.trainable = False

    # Get base model output
    base_model_ouput = base_model.output

    # Adding our own layer
    x = GlobalAveragePooling2D()(base_model_ouput)
    # Adding fully connected layer
    x = Dense(512, activation="relu")(x)
    x = Dense(num_classes, activation="softmax", name="fcnew")(x)

    model = Model(inputs=base_model.input, outputs=x)
    return model

In [6]:
# Get the model
model = get_model()
# Compile it
model.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
# Summary of model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, None,      │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, None,      │      9,472 │ conv1_pad[0][0]   │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, None,      │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, None,      │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, None,      │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, None,      │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, None,      │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, None,      │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, None,      │          0 │ conv2_block1_1_b… │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, None,      │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, None,      │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, None,      │          0 │ conv2_block1_2_b… │
│ (Activation)        │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, None,      │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, None,      │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, None,      │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ None, 256)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, None,      │      1,024 │ conv2_block1_3_c

 Total params: 24,639,365 (93.99 MB)

 Trainable params: 1,051,653 (4.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [7]:
# Defining the imagedatagenerator for train and test image for pre-processing
# We don't give horizonal_flip or other preprocessing for validation data generator

image_size = 224
batch_size = 64

train_data_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
valid_data_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator = train_data_gen.flow_from_directory(
    training_dir,
    (image_size, image_size),
    batch_size=batch_size,
    class_mode="categorical",
)
valid_generator = valid_data_gen.flow_from_directory(
    testing_dir,
    (image_size, image_size),
    batch_size=batch_size,
    class_mode="categorical",
)

Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


In [8]:
# Training the fully conncected layer for initial epochs
epochs = 5

# Training the model

model.fit(
    train_generator,
    steps_per_epoch=train_generator.n // batch_size,
    validation_data=valid_generator,
    validation_steps=valid_generator.n // batch_size,
    epochs=epochs,
    verbose=1,
)

C:\Users\ak007\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: The PyDataset has length 0

In [ ]:
# More fine tuning the model
# Training the model after 150 layers
# Generally ResNet is good at extracting lower level features so we are not fine tuning initial layers
epochs = 10

split_at = 140
for layer in model.layers[:split_at]:
    layer.trainable = False
for layer in model.layers[split_at:]:
    layer.trainable = True

model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

# Choosing lower learning rate for fine-tuning
# learning rate is generally 10-1000 times lower than normal learning rate, if we are fine tuning the initial layers
sgd = optimizers.SGD(lr=0.001, decay=1e-6, momentum=0.9, nesterov=True)

model.compile(loss="categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

model.fit_generator(
    train_generator,
    steps_per_epoch=train_generator.n // batch_size,
    validation_data=valid_generator,
    validation_steps=valid_generator.n // batch_size,
    epochs=epochs,
    verbose=1,
)

Epoch 1/10
54/54 [==============================] - 60s 1s/step - loss: 0.2540 - acc: 0.9230 - val_loss: 0.3037 - val_acc: 0.8942
Epoch 2/10
54/54 [==============================] - 58s 1s/step - loss: 0.2345 - acc: 0.9217 - val_loss: 0.2753 - val_acc: 0.9014
Epoch 3/10
54/54 [==============================] - 58s 1s/step - loss: 0.1598 - acc: 0.9537 - val_loss: 0.2552 - val_acc: 0.9135
Epoch 4/10
54/54 [==============================] - 57s 1s/step - loss: 0.1410 - acc: 0.9543 - val_loss: 0.2575 - val_acc: 0.9111
Epoch 5/10
54/54 [==============================] - 58s 1s/step - loss: 0.1146 - acc: 0.9687 - val_loss: 0.2363 - val_acc: 0.9195
Epoch 6/10
54/54 [==============================] - 57s 1s/step - loss: 0.1036 - acc: 0.9654 - val_loss: 0.2373 - val_acc: 0.9219
Epoch 7/10
54/54 [==============================] - 56s 1s/step - loss: 0.0805 - acc: 0.9797 - val_loss: 0.2345 - val_acc: 0.9171
Epoch 8/10
54/54 [==============================] - 56s 1s/step - loss: 0.0974 - acc: 0.96

In [ ]:
print("Training complete")

Training complete


In [2]:
import numpy as np

A = np.array([[1, 5], [-2, 4], [8, 3]])
B = A.T
C = A @ B
C

array([[26, 18, 23],
       [18, 20, -4],
       [23, -4, 73]])